# Restrict to Topic Guardrail

**What is topic restriction?** Keeping an application to the subjects it's meant to cover, and refusing everything else.

**How this guardrail helps:** a RAG assistant for one subject shouldn't spend LLM calls on, or give answers about, unrelated subjects. The Restrict to Topic validator classifies the text against your topics. Validation fails if it finds an invalid topic, or if it finds none of the valid ones.

**One-time setup** (already done in `myenv`, the kernel for this notebook):
```bash
pip install guardrails-ai guardrails-ai-restrict-to-topic
```

The first run downloads the classifier model, `facebook/bart-large-mnli`.

In [1]:
import warnings

# silence harmless warnings from tqdm and guardrails' sync fallback in Jupyter
warnings.filterwarnings("ignore", message="IProgress not found")
warnings.filterwarnings("ignore", message="Could not obtain an event loop")

from dotenv import load_dotenv
from guardrails import Guard  ## the main wrapper for Guardrails
from guardrails.errors import ValidationError
from guardrails_ai.restricttotopic import RestrictToTopic  ## note: no underscores in the module name

# load the env vars
load_dotenv()

True

## 1. Create the validator and the guard

`disable_llm=True` uses only the local classifier. Without it, the validator also calls OpenAI on every check, and fails with `OpenAIError: Missing credentials` if `OPENAI_API_KEY` isn't set.

In [2]:
valid_topics = ["machine learning", "artificial intelligence", "data science"]
invalid_topics = ["sports", "politics", "entertainment"]

# create the validator
validator = RestrictToTopic(valid_topics=valid_topics,
                            invalid_topics=invalid_topics,
                            disable_llm=True,
                            on_fail="exception")

# create guard and add the validator to the guard
guard = Guard().use(validator)
guard.configure(allow_metrics_collection=False)

Device set to use cpu


## 2. Validate user input

In [3]:
user_input_1 = "Can you tell me what is Linear Regression?"
user_input_2 = "Who won the IPL final in 2026?"

response_1 = guard.validate(user_input_1)

print(f"Validation Result: {response_1.validation_passed}")
print(f"Raw User Input: {response_1.raw_llm_output}")
print(f"Validated Output: {response_1.validated_output}")

Validation Result: True
Raw User Input: Can you tell me what is Linear Regression?
Validated Output: Can you tell me what is Linear Regression?


In [4]:
# on_fail="exception" -> an off-topic input raises a ValidationError
try:
    response_2 = guard.validate(user_input_2)
except ValidationError as e:
    print(f"Blocked: {e}")

Blocked: Validation failed for field with errors: Invalid topics found: ['sports', 'entertainment']


## 3. `on_fail="refrain"` — return no output instead of raising

In [5]:
refrain_guard = Guard().use(RestrictToTopic(valid_topics=valid_topics,
                                            invalid_topics=invalid_topics,
                                            disable_llm=True,
                                            on_fail="refrain"))
refrain_guard.configure(allow_metrics_collection=False)

response_3 = refrain_guard.validate(user_input_2)

print(f"Validation Result: {response_3.validation_passed}")
print(f"Validated Output: {response_3.validated_output}")
print(f"Failure Reason: {response_3.validation_summaries[0].failure_reason}")

Device set to use cpu


Validation Result: False
Validated Output: None
Failure Reason: Invalid topics found: ['sports', 'entertainment']


## 4. What it can get wrong

The classifier only finds a topic when its score is above `zero_shot_threshold` (default `0.5`), and that doesn't always match what you'd expect:

- **On-topic questions can be blocked:** *"What does overfitting mean?"* is machine learning, but no valid topic scores above `0.5`. Lowering `zero_shot_threshold` may help, but test that it doesn't let off-topic text through.
- **Any invalid topic wins:** the football prediction question scores high on machine learning, but it's blocked because it's also about sports.
- **Short messages are unreliable:** a simple *"hi"* is labelled `entertainment`, so handle greetings before this check.
- **Loosely related questions can pass:** the linked list question is about programming, but it counts as data science.

Choose topic names carefully and test them on real questions from your application.

In [6]:
examples = [
    "What does overfitting mean?",
    "How is machine learning used to predict football match results?",
    "hi",
    "How do I reverse a linked list in Python?",
    "What is a good recipe for pasta?",
]

for text in examples:
    try:
        guard.validate(text)
        print(f"{text} -> passed")
    except ValidationError as e:
        print(f"{text} -> blocked: {str(e).split('errors: ')[-1]}")

What does overfitting mean? -> blocked: No valid topic was found.
How is machine learning used to predict football match results? -> blocked: Invalid topics found: ['sports']
hi -> blocked: Invalid topics found: ['entertainment']
How do I reverse a linked list in Python? -> passed
What is a good recipe for pasta? -> blocked: No valid topic was found.
